In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("ibm_jobs_with_unsupervised_results.csv")

print("Dataset shape:", df.shape)


target = "log_mid_salary"
df = df.dropna(subset=[target])

y = df[target]

leakage_cols = [
    "log_mid_salary", "mid_salary", "min_salary", "max_salary",
    "salary_min", "salary_max", "salary_mid"
]

X = df.drop(columns=[col for col in leakage_cols if col in df.columns])

text_cols = ["text_all", "job_description", "description"]
X = X.drop(columns=[col for col in text_cols if col in X.columns], errors="ignore")

numeric_features = X.select_dtypes(include=["int64", "float64", "bool"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

print("Numeric:", len(numeric_features), "Categorical:", len(categorical_features))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot)
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

def evaluate_model(model, name):
    
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    cv_mse = -cross_val_score(
        pipe,
        X_train,
        y_train,
        scoring="neg_mean_squared_error",
        cv=5,
        n_jobs=-1
    )
    
    cv_rmse = np.sqrt(cv_mse).mean()
    
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
    mae_log = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    y_test_d = np.exp(y_test)
    y_pred_d = np.exp(y_pred)
    
    rmse_d = np.sqrt(mean_squared_error(y_test_d, y_pred_d))
    mae_d = mean_absolute_error(y_test_d, y_pred_d)
    
    return {
        "Model": name,
        "CV RMSE": cv_rmse,
        "Test RMSE": rmse_d,
        "MAE": mae_d,
        "R2": r2
    }


models = [
    ("Ridge", Ridge(alpha=0.8)),
    ("Random Forest", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ("Gradient Boosting", GradientBoostingRegressor(random_state=42))
]

results = []

for name, model in models:
    print("Running:", name)
    results.append(evaluate_model(model, name))


results_df = pd.DataFrame(results)

results_df["Interpretability"] = results_df["Model"].map({
    "Ridge": "High",
    "Random Forest": "Medium",
    "Gradient Boosting": "Low"
})

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

display(results_df)

results_df.to_csv("model_results.csv", index=False)

print("Saved → model_results.csv")

import matplotlib.pyplot as plt

models = ["Ridge", "Random Forest", "Gradient Boosting"]

rmse = [11958.58, 5394.46, 2843.02]
r2 = [0.9630, 0.9951, 0.9983]

plt.figure()
plt.bar(models, rmse)
plt.title("Model Comparison - RMSE")
plt.ylabel("Test RMSE (USD)")
plt.savefig("rmse_comparison.png", dpi=300)
plt.close()

plt.figure()
plt.bar(models, r2)
plt.title("Model Comparison - R2")
plt.ylabel("Test R^2")
plt.savefig("r2_comparison.png", dpi=300)
plt.close()

Dataset shape: (469, 82)
Numeric: 64 Categorical: 13
Running: Ridge
Running: Random Forest
Running: Gradient Boosting


,Model,CV RMSE,Test RMSE,MAE,R2,Interpretability
0,Ridge,0.0894,11958.5833,7431.9666,0.9630,High
1,Random Forest,0.0355,5394.4637,1633.8126,0.9951,Medium
2,Gradient Boosting,0.0232,2843.0156,1119.4361,0.9983,Low


Saved → model_results.csv
